In [1]:
import os
import pandas as pd
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
original_files = home + "Other/"
h5n1_files = home + "Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--08-15-2025_Antarctica_North_America_South_America/"
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# os.chdir(references)
# states = pd.read_csv("states_ref.csv")

original_file_fastas = {}
for dirpath, dirs, files in os.walk(original_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".txt" in file_name:
            # print(file_name)
            fasta_file = df_from_fasta(file_name) # Convert fasta file to dataframe
            # print(fasta_file)
            isolates = fasta_file["full_header"].apply(lambda x: x.split("/")[3])
            header0 = fasta_file["full_header"].apply(lambda x: x.split(".")[0] + "|")
            header1 = fasta_file["full_header"].apply(lambda x: x.split(")")[0] + ")") 
            header2 = header1.apply(lambda x: x.replace(x.split("(")[0], ""))
            fasta_file["full_header"] = header0 + header2
            fasta_file["full_header"] = fasta_file["full_header"].apply(lambda x: x.replace("(", "", 1))
            print(fasta_file["full_header"])
            fasta_file["isolate"] = isolates
            # print(isolates)
            original_file_fastas[file_name] = fasta_file

h5n1_file_fastas = {}
for dirpath, dirs, files in os.walk(h5n1_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        fasta_file = df_from_fasta(file_name) # Convert fasta file to dataframe
        # print(fasta_file)
        isolates = fasta_file["full_header"].apply(lambda x: x.split("/")[3])
        fasta_file["isolate"] = isolates
        h5n1_file_fastas[file_name] = fasta_file

for key in original_file_fastas:
    og_fasta = original_file_fastas[key]
    for h5n1_key in h5n1_file_fastas:
        h5n1_fasta = h5n1_file_fastas[h5n1_key]
        common = og_fasta.merge(h5n1_fasta, on="isolate", how="inner")
        if len(common) > 0:
            common["full_header"] = common["full_header_y"]
            common["sequence"] = common["sequence_x"]
            common = common[["full_header", "sequence"]]
            # print(common)
            right = common.merge(og_fasta, on="sequence", how="right")
            
            right["full_header"] = right["full_header_x"].fillna(right["full_header_y"])
            right = right[["full_header", "sequence"]]
            print(right)
            df_to_fasta(right, "", key[:-4] + "_edit.fasta")
        else:
            df_to_fasta(og_fasta, "", key[:-4] + "_none_found.fasta")

0     >PQ719222|A/Turkey Vulture/NY/22-019921-001-or...
1     >OQ963697|A/Royal Tern/Florida/22-005590-002/2...
2     >PQ719230|A/Turkey Vulture/NY/22-019922-001-or...
3     >OQ959425|A/Bald Eagle/Minnesota/22-011695-001...
4     >PQ717094|A/Ross's goose/MS/22-038267-001-orig...
5     >OQ961825|A/Herring Gull/Michigan/22-012056-00...
6     >OQ960921|A/Duck/Florida/22-006447-001/2022(H5N1)
7     >OQ963177|A/Red-tailed Hawk/Wisconsin/22-00941...
8     >OQ961769|A/Gull/Florida/22-009801-002/2022(H5N1)
9     >OQ958929|A/Backyard bird/Wisconsin/22-010325-...
10    >OQ963041|A/Red-tailed Hawk/New York/22-011864...
11    >OQ958921|A/Backyard bird/Wisconsin/22-010325-...
12    >OQ962009|A/Lesser Scaup/Georgia/22-006543-001...
13    >OQ962041|A/Lesser Scaup/Georgia/22-006543-006...
14    >OQ959185|A/Bald Eagle/Georgia/22-008982-003/2...
15    >OQ959273|A/Bald Eagle/Michigan/22-011517-002/...
16    >OQ958785|A/Backyard bird/New York/22-010200-0...
17    >OQ958489|A/Backyard bird/Michigan/22-0115